# Data Analysis Assignment: EDA & Data Cleaning
**Datasets:** 1) Loan Eligibility  2) Adult Census Income  3) Hotel Booking  
**Tools:** Python (Pandas, NumPy, Matplotlib, Seaborn)

Har dataset par yeh steps kiye gaye hain:
1. Data load karna aur pehli nazar (head, shape, info, describe)
2. Missing values aur duplicates check karna
3. Data cleaning
4. Outliers dekhna aur handle karna
5. Har column ka graph (aur target ke saath relation)
6. Correlation aur nateeja (findings)

**Note:** `loan-train.csv`, `adult.csv` aur `hotel_booking.csv` is notebook wale folder mein honi chahiye.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

---
# Part 1: Loan Eligibility Dataset
**Goal:** Pata karna ke kin cheezon ki wajah se loan approve (`Y`) ya reject (`N`) hota hai.

## 1.1 Data load karna aur pehli nazar

In [ ]:
loan = pd.read_csv("loan-train.csv")
print("Shape:", loan.shape)
loan.head()

In [ ]:
# Columns ki information (naam, null count, data type)
loan.info()

In [ ]:
# Numeric columns ka statistical summary
loan.describe()

In [ ]:
# Text columns ka summary
loan.describe(include="object")

## 1.2 Missing values aur duplicates

In [ ]:
# Har column mein kitni values missing hain
print(loan.isnull().sum())

In [ ]:
# Missing values ka graph (heatmap mein light rang = missing)
plt.figure(figsize=(10, 4))
sns.heatmap(loan.isnull(), cbar=False, yticklabels=False)
plt.title("Missing values map")
plt.show()

In [ ]:
# Duplicate rows
print("Poori duplicate rows:", loan.duplicated().sum())
print("Duplicate Loan_ID:", loan["Loan_ID"].duplicated().sum())
# Loan_ID repeat hui hai magar baaki columns alag hain, is liye rows delete nahi ki jayengi

In [ ]:
# Har categorical column ki unique values (spelling ki ghalti check karne ke liye)
for col in ["Gender", "Married", "Dependents", "Education", "Self_Employed", "Property_Area", "Loan_Status"]:
    print(col, "->", loan[col].dropna().unique())

## 1.3 Data Cleaning
- Text se extra spaces hataye
- `Dependents` ki value `3+` ko `3` banaya
- Text columns ke missing values **mode** (sab se zyada aane wali value) se bhare
- `LoanAmount` ke missing values **median** se bhare
- `Loan_Amount_Term` aur `Credit_History` mode se bhare

In [ ]:
loan_clean = loan.copy()

# 1) Extra spaces hatana
for col in loan_clean.select_dtypes(exclude="number").columns:
    loan_clean[col] = loan_clean[col].str.strip()

# 2) Dependents: 3+ ko 3 banana
loan_clean["Dependents"] = loan_clean["Dependents"].replace("3+", "3")

# 3) Text columns ke missing values mode se
for col in ["Gender", "Married", "Dependents", "Self_Employed"]:
    loan_clean[col] = loan_clean[col].fillna(loan_clean[col].mode()[0])

# 4) Numeric columns ke missing values
loan_clean["LoanAmount"] = loan_clean["LoanAmount"].fillna(loan_clean["LoanAmount"].median())
loan_clean["Loan_Amount_Term"] = loan_clean["Loan_Amount_Term"].fillna(loan_clean["Loan_Amount_Term"].mode()[0])
loan_clean["Credit_History"] = loan_clean["Credit_History"].fillna(loan_clean["Credit_History"].mode()[0])

# 5) Data types theek karna
loan_clean["Dependents"] = loan_clean["Dependents"].astype(int)
loan_clean["Credit_History"] = loan_clean["Credit_History"].astype(int)
loan_clean["Loan_Amount_Term"] = loan_clean["Loan_Amount_Term"].astype(int)

print("Cleaning ke baad missing values:", loan_clean.isnull().sum().sum())

In [ ]:
# Cleaning ke baad dobara check
print(loan_clean.isnull().sum())
loan_clean.info()

## 1.4 Outliers
`ApplicantIncome`, `CoapplicantIncome` aur `LoanAmount` mein bohat badi values hain. Pehle boxplot dekhte hain, phir IQR method se limit se bahar ki values ko limit par laga denge (rows delete nahi karenge).

In [ ]:
num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]

plt.figure(figsize=(15, 3.5))
for i, col in enumerate(num_cols):
    plt.subplot(1, 3, i + 1)
    sns.boxplot(x=loan_clean[col], color="lightcoral")
    plt.title(col + " (pehle)")
plt.tight_layout()
plt.show()

In [ ]:
# IQR method: Q1 - 1.5*IQR se neeche aur Q3 + 1.5*IQR se upar outlier hai
for col in num_cols:
    q1 = loan_clean[col].quantile(0.25)
    q3 = loan_clean[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    count = ((loan_clean[col] < low) | (loan_clean[col] > high)).sum()
    print(col, "| outliers:", count, "| lower:", round(low, 1), "| upper:", round(high, 1))
    # limit se bahar ki values ko limit par set karna
    loan_clean[col] = loan_clean[col].clip(low, high)

# Naya column: dono ki income ka total
loan_clean["TotalIncome"] = loan_clean["ApplicantIncome"] + loan_clean["CoapplicantIncome"]

## 1.5 Har column ka analysis
Pehle target `Loan_Status`, phir categorical columns (counts aur har category mein approval %), phir numeric columns (histogram aur boxplot).

In [ ]:
# Target column: Loan_Status
plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1)
sns.countplot(data=loan_clean, x="Loan_Status")
plt.title("Loan_Status counts")
plt.subplot(1, 2, 2)
loan_clean["Loan_Status"].value_counts().plot.pie(autopct="%1.1f%%")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Loan_ID sirf pehchan (identifier) hai, is ka analysis nahi karte
print("Total rows:", len(loan_clean), "| Unique Loan_ID:", loan_clean["Loan_ID"].nunique())

In [ ]:
# Categorical columns: left = count, right = Loan_Status ka percentage
cat_cols = ["Gender", "Married", "Dependents", "Education", "Self_Employed",
            "Loan_Amount_Term", "Credit_History", "Property_Area"]

for col in cat_cols:
    plt.figure(figsize=(12, 3.5))
    plt.subplot(1, 2, 1)
    sns.countplot(data=loan_clean, x=col)
    plt.title(col + ": counts")
    plt.xticks(rotation=45)

    plt.subplot(1, 2, 2)
    percent = pd.crosstab(loan_clean[col], loan_clean["Loan_Status"], normalize="index") * 100
    percent.plot(kind="bar", stacked=True, ax=plt.gca())
    plt.title(col + " vs Loan_Status (%)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    print(loan_clean[col].value_counts().to_dict())

In [ ]:
# Numeric columns: histogram | boxplot | Loan_Status ke saath boxplot
for col in ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "TotalIncome"]:
    plt.figure(figsize=(14, 3.5))
    plt.subplot(1, 3, 1)
    sns.histplot(loan_clean[col], kde=True)
    plt.title(col + ": distribution")
    plt.subplot(1, 3, 2)
    sns.boxplot(x=loan_clean[col], color="lightcoral")
    plt.title(col + ": boxplot")
    plt.subplot(1, 3, 3)
    sns.boxplot(data=loan_clean, x="Loan_Status", y=col, showfliers=False)
    plt.title(col + " vs Loan_Status")
    plt.tight_layout()
    plt.show()
    print(loan_clean[col].describe().round(2).to_dict())

## 1.6 Correlation

In [ ]:
loan_corr = loan_clean.copy()
loan_corr["Loan_Status_num"] = (loan_corr["Loan_Status"] == "Y").astype(int)   # Y = 1, N = 0

cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term",
        "Credit_History", "Dependents", "TotalIncome", "Loan_Status_num"]
plt.figure(figsize=(8, 6))
sns.heatmap(loan_corr[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap (Loan)")
plt.show()

In [ ]:
# Do columns ka milakar approval rate
plt.figure(figsize=(7, 4))
rate = loan_corr.pivot_table(index="Property_Area", columns="Credit_History", values="Loan_Status_num", aggfunc="mean")
sns.heatmap(rate, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Approval rate: Property_Area x Credit_History")
plt.show()

## 1.7 Summary

In [ ]:
print("Rows:", len(loan_clean))
print("Overall approval rate: %.1f%%" % (loan_corr["Loan_Status_num"].mean() * 100))
print("Credit_History ke hisaab se approval %:")
print((loan_corr.groupby("Credit_History")["Loan_Status_num"].mean() * 100).round(1))
print("Property_Area ke hisaab se approval %:")
print((loan_corr.groupby("Property_Area")["Loan_Status_num"].mean() * 100).round(1))

# Clean data save karna
loan_clean.to_csv("cleaned_loan_dataset.csv", index=False)

**Key findings (Loan):**
- `Gender, Married, Dependents, Self_Employed, LoanAmount, Loan_Amount_Term, Credit_History` mein missing values thin; mode/median se bhar diye.
- `Loan_ID` repeat hui hai lekin rows alag hain, is liye rows delete nahi ki.
- Income aur LoanAmount mein outliers the; IQR limit par cap kiye.
- **Credit_History sab se zyada asar karne wala column hai:** jin ka credit history 1 hai unka approval rate zyada hai.
- `Property_Area`, `Married` aur `Education` ka bhi approval par thora asar hai; `Gender` ka asar kam hai.

---
# Part 2: Adult Census Income Dataset
**Goal:** Pata karna ke kin cheezon se income `>50K` hoti hai.

## 2.1 Data load karna aur pehli nazar

In [ ]:
adult = pd.read_csv("adult.csv")
print("Shape:", adult.shape)
adult.head()

In [ ]:
adult.info()

In [ ]:
adult.describe()

In [ ]:
adult.describe(include="object")

## 2.2 Missing values aur duplicates
Is dataset mein missing values `?` likh kar di gayi hain, is liye `isnull()` unhein nahi pakadta.

In [ ]:
print("isnull() ka total:", adult.isnull().sum().sum(), "(yeh galat tasalli deta hai)")

# '?' kitni jagah hai
for col in adult.select_dtypes(exclude="number").columns:
    count = (adult[col].str.strip() == "?").sum()
    if count > 0:
        print(col, "-> '?' count:", count)

In [ ]:
print("Duplicate rows:", adult.duplicated().sum())

# Categorical columns ki unique values
for col in ["workclass", "marital-status", "occupation", "relationship", "race", "gender", "income"]:
    print(col, "->", adult[col].unique())

## 2.3 Data Cleaning
- Text se extra spaces hataye
- `?` ko `NaN` banaya, phir `workclass`, `occupation`, `native-country` ko **mode** se bhara
- Duplicate rows delete ki
- `income` ka number wala column `income_bin` banaya (`>50K` = 1, `<=50K` = 0)

In [ ]:
adult_clean = adult.copy()

# 1) Extra spaces hatana
for col in adult_clean.select_dtypes(exclude="number").columns:
    adult_clean[col] = adult_clean[col].str.strip()

# 2) '?' ko NaN banana
adult_clean = adult_clean.replace("?", np.nan)
print(adult_clean.isnull().sum()[adult_clean.isnull().sum() > 0])

# 3) Missing values mode se bharna
for col in ["workclass", "occupation", "native-country"]:
    adult_clean[col] = adult_clean[col].fillna(adult_clean[col].mode()[0])

# 4) Duplicate rows delete karna
before = len(adult_clean)
adult_clean = adult_clean.drop_duplicates().reset_index(drop=True)
print("Duplicate delete hui:", before - len(adult_clean), "| Baqi rows:", len(adult_clean))

# 5) income ka number wala column
adult_clean["income_bin"] = (adult_clean["income"] == ">50K").astype(int)

print("Missing values baqi:", adult_clean.isnull().sum().sum())

## 2.4 Outliers
`capital-gain` aur `capital-loss` mein zyada tar values 0 hain (aur `99999` ek khaas value hai), is liye in ko cap nahi kiya. `age`, `fnlwgt` aur `hours-per-week` ko IQR se cap kiya.

In [ ]:
num_cols = ["age", "fnlwgt", "educational-num", "capital-gain", "capital-loss", "hours-per-week"]

plt.figure(figsize=(15, 6))
for i, col in enumerate(num_cols):
    plt.subplot(2, 3, i + 1)
    sns.boxplot(x=adult_clean[col], color="lightcoral")
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Outliers ginna
for col in num_cols:
    q1 = adult_clean[col].quantile(0.25)
    q3 = adult_clean[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    count = ((adult_clean[col] < low) | (adult_clean[col] > high)).sum()
    print(col, "| outliers:", count)

print("capital-gain == 99999:", (adult_clean["capital-gain"] == 99999).sum())
print("capital-gain zero %:", round((adult_clean["capital-gain"] == 0).mean() * 100, 1))
print("capital-loss zero %:", round((adult_clean["capital-loss"] == 0).mean() * 100, 1))

In [ ]:
# Sirf age, fnlwgt, hours-per-week ko cap karna
for col in ["age", "fnlwgt", "hours-per-week"]:
    q1 = adult_clean[col].quantile(0.25)
    q3 = adult_clean[col].quantile(0.75)
    iqr = q3 - q1
    adult_clean[col] = adult_clean[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

# Naye columns: capital gain/loss hai ya nahi
adult_clean["has_capital_gain"] = (adult_clean["capital-gain"] > 0).astype(int)
adult_clean["has_capital_loss"] = (adult_clean["capital-loss"] > 0).astype(int)

## 2.5 Har column ka analysis

In [ ]:
# Target column: income
plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1)
sns.countplot(data=adult_clean, x="income")
plt.title("income counts")
plt.subplot(1, 2, 2)
adult_clean["income"].value_counts().plot.pie(autopct="%1.1f%%")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Categorical columns: left = count, right = income ka percentage
cat_cols = ["workclass", "education", "marital-status", "occupation", "relationship", "race", "gender"]

for col in cat_cols:
    order = adult_clean[col].value_counts().index
    plt.figure(figsize=(14, 4))
    plt.subplot(1, 2, 1)
    sns.countplot(data=adult_clean, y=col, order=order)
    plt.title(col + ": counts")

    plt.subplot(1, 2, 2)
    percent = pd.crosstab(adult_clean[col], adult_clean["income"], normalize="index") * 100
    percent.loc[order].plot(kind="barh", stacked=True, ax=plt.gca())
    plt.title(col + " vs income (%)")
    plt.tight_layout()
    plt.show()
    print(adult_clean[col].value_counts().to_dict())

In [ ]:
# native-country mein 40 se zyada countries hain, is liye top 10 dikhate hain
top10 = adult_clean["native-country"].value_counts().head(10).index
temp = adult_clean[adult_clean["native-country"].isin(top10)]

plt.figure(figsize=(14, 4))
plt.subplot(1, 2, 1)
sns.countplot(data=temp, y="native-country", order=top10)
plt.title("native-country: top 10")
plt.subplot(1, 2, 2)
percent = pd.crosstab(temp["native-country"], temp["income"], normalize="index") * 100
percent.loc[top10].plot(kind="barh", stacked=True, ax=plt.gca())
plt.title("native-country vs income (%)")
plt.tight_layout()
plt.show()
print(adult_clean["native-country"].value_counts(normalize=True).head(3).mul(100).round(1))

In [ ]:
# Numeric columns: histogram | boxplot | income ke saath boxplot
for col in ["age", "fnlwgt", "educational-num", "capital-gain", "capital-loss", "hours-per-week"]:
    plt.figure(figsize=(14, 3.5))
    plt.subplot(1, 3, 1)
    sns.histplot(adult_clean[col], kde=True)
    plt.title(col + ": distribution")
    plt.subplot(1, 3, 2)
    sns.boxplot(x=adult_clean[col], color="lightcoral")
    plt.title(col + ": boxplot")
    plt.subplot(1, 3, 3)
    sns.boxplot(data=adult_clean, x="income", y=col, showfliers=False)
    plt.title(col + " vs income")
    plt.tight_layout()
    plt.show()
    print(adult_clean[col].describe().round(2).to_dict())

In [ ]:
# capital-gain / capital-loss: sirf non-zero values ka histogram
plt.figure(figsize=(12, 3.5))
plt.subplot(1, 2, 1)
sns.histplot(adult_clean[adult_clean["capital-gain"] > 0]["capital-gain"], bins=40)
plt.title("capital-gain (sirf non-zero)")
plt.subplot(1, 2, 2)
sns.histplot(adult_clean[adult_clean["capital-loss"] > 0]["capital-loss"], bins=40)
plt.title("capital-loss (sirf non-zero)")
plt.tight_layout()
plt.show()

## 2.6 Correlation aur multivariate

In [ ]:
cols = ["age", "fnlwgt", "educational-num", "capital-gain", "capital-loss", "hours-per-week", "income_bin"]
plt.figure(figsize=(8, 6))
sns.heatmap(adult_clean[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap (Adult)")
plt.show()

In [ ]:
# Gender/race aur age group ke hisaab se >50K ka chance
adult_clean["age_group"] = pd.cut(adult_clean["age"], [16, 25, 35, 45, 55, 65, 100],
                                  labels=["<=25", "26-35", "36-45", "46-55", "56-65", "65+"])
plt.figure(figsize=(15, 4))
plt.subplot(1, 2, 1)
sns.barplot(data=adult_clean, x="gender", y="income_bin", hue="race")
plt.title("P(income>50K) by gender and race")
plt.subplot(1, 2, 2)
sns.barplot(data=adult_clean, x="age_group", y="income_bin", hue="gender")
plt.title("P(income>50K) by age group and gender")
plt.tight_layout()
plt.show()

## 2.7 Summary

In [ ]:
print("Rows after cleaning:", len(adult_clean))
print(">50K share: %.1f%%" % (adult_clean["income_bin"].mean() * 100))
print(">50K % by gender:")
print((adult_clean.groupby("gender")["income_bin"].mean() * 100).round(1))
print("Top 3 occupations by >50K %:")
print((adult_clean.groupby("occupation")["income_bin"].mean() * 100).round(1).sort_values(ascending=False).head(3))

# Clean data save karna
adult_clean.to_csv("cleaned_adult_dataset.csv", index=False)

**Key findings (Adult):**
- Missing values `?` ki shakal mein thin (`workclass`, `occupation`, `native-country`); `NaN` bana kar mode se bhare. Duplicate rows delete ki.
- Dataset imbalanced hai: `<=50K` wale zyada hain (lagbhag 76%).
- Zyada education, 35 se 55 saal ki age aur zyada hours-per-week ke saath `>50K` ka chance barhta hai.
- Married log (`Married-civ-spouse`, `Husband`/`Wife`) aur `Exec-managerial` / `Prof-specialty` occupations mein `>50K` zyada hai.
- `capital-gain` aur `capital-loss` zyada tar zero hain; `fnlwgt` sirf sampling weight hai, income se is ka rishta kamzor hai.
- Male aur female ke beech income ka farq nazar aata hai.

---
# Part 3: Hotel Booking Dataset
**Goal:** Booking cancel hone (`is_canceled`) aur booking ke behaviour ko samajhna.

## 3.1 Data load karna aur pehli nazar

In [ ]:
hotel = pd.read_csv("hotel_booking.csv")
print("Shape:", hotel.shape)
hotel.head()

In [ ]:
hotel.info()

In [ ]:
hotel.describe()

In [ ]:
hotel.describe(include="object")

## 3.2 Missing values aur duplicates

In [ ]:
missing = hotel.isnull().sum()
missing = missing[missing > 0]
print(missing)

plt.figure(figsize=(7, 3))
sns.barplot(x=missing.values, y=missing.index)
plt.title("Missing values per column")
plt.show()

In [ ]:
print("Duplicate rows:", hotel.duplicated().sum())

# Galat / ajeeb values check
print("meal 'Undefined':", (hotel["meal"] == "Undefined").sum())
print("market_segment 'Undefined':", (hotel["market_segment"] == "Undefined").sum())
print("distribution_channel 'Undefined':", (hotel["distribution_channel"] == "Undefined").sum())
print("0 guests wali rows:", ((hotel["adults"] + hotel["children"].fillna(0) + hotel["babies"]) == 0).sum())
print("0 nights wali rows:", ((hotel["stays_in_weekend_nights"] + hotel["stays_in_week_nights"]) == 0).sum())
print("Negative adr rows:", (hotel["adr"] < 0).sum(), "| adr max:", hotel["adr"].max())

## 3.3 Data Cleaning
- `children` (4 missing) -> 0; `country` (488 missing) -> `"Unknown"`
- `agent` aur `company` ke missing ka matlab "koi agent/company nahi" hai, is liye 0 bhara
- `Undefined` meal ko `SC` (no meal package) bana diya
- Exact duplicate rows nahi hain (har row ka name/email/phone alag hai), is liye delete nahi ki
- 0 guests wali bookings, negative `adr` aur bohat zyada `adr` (5400) wali rows hata di
- Date columns ko date type mein badla
- Naye columns: `total_nights`, `total_guests`

In [ ]:
hotel_clean = hotel.copy()

hotel_clean["children"] = hotel_clean["children"].fillna(0).astype(int)
hotel_clean["country"] = hotel_clean["country"].fillna("Unknown")
hotel_clean["agent"] = hotel_clean["agent"].fillna(0).astype(int)
hotel_clean["company"] = hotel_clean["company"].fillna(0).astype(int)
hotel_clean["meal"] = hotel_clean["meal"].replace("Undefined", "SC")

# Email ka domain (personal columns drop karne se pehle)
hotel_clean["email_domain"] = hotel_clean["email"].str.split("@").str[1]

In [ ]:
# Date columns
months = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
month_number = {name: i + 1 for i, name in enumerate(months)}
hotel_clean["arrival_month_num"] = hotel_clean["arrival_date_month"].map(month_number)
hotel_clean["arrival_date"] = pd.to_datetime(dict(year=hotel_clean["arrival_date_year"],
                                                  month=hotel_clean["arrival_month_num"],
                                                  day=hotel_clean["arrival_date_day_of_month"]))
hotel_clean["reservation_status_date"] = pd.to_datetime(hotel_clean["reservation_status_date"])

# Naye columns
hotel_clean["total_nights"] = hotel_clean["stays_in_weekend_nights"] + hotel_clean["stays_in_week_nights"]
hotel_clean["total_guests"] = hotel_clean["adults"] + hotel_clean["children"] + hotel_clean["babies"]

In [ ]:
# Galat rows hatana
start = len(hotel_clean)
hotel_clean = hotel_clean[hotel_clean["total_guests"] > 0]
hotel_clean = hotel_clean[(hotel_clean["adr"] >= 0) & (hotel_clean["adr"] < 5000)]
hotel_clean = hotel_clean.reset_index(drop=True)
print("Pehle:", start, "| Baad mein:", len(hotel_clean))
print("Missing values baqi:", hotel_clean.isnull().sum().sum())

## 3.4 Outliers
Continuous columns (`lead_time`, `adr`, nights) ko IQR se cap kiya. Count wale columns (`children`, `babies`, `previous_cancellations` etc.) zyada tar 0 hain aur 0 se upar ki values asli behaviour hain, is liye unhein cap nahi kiya.

In [ ]:
plt.figure(figsize=(15, 6))
for i, col in enumerate(["lead_time", "adr", "total_nights", "adults", "days_in_waiting_list", "booking_changes"]):
    plt.subplot(2, 3, i + 1)
    sns.boxplot(x=hotel_clean[col], color="lightcoral")
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
for col in ["lead_time", "adr", "stays_in_week_nights", "stays_in_weekend_nights", "total_nights"]:
    q1 = hotel_clean[col].quantile(0.25)
    q3 = hotel_clean[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    count = ((hotel_clean[col] < low) | (hotel_clean[col] > high)).sum()
    print(col, "| outliers:", count, "| upper limit:", round(high, 1))
    hotel_clean[col] = hotel_clean[col].clip(low, high)

# adults ki 55 jaisi values galti ya bade group hain, is liye max 4 rakha
hotel_clean["adults"] = hotel_clean["adults"].clip(upper=4)

## 3.5 Har column ka analysis

In [ ]:
# Target column: is_canceled
plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1)
sns.countplot(data=hotel_clean, x="is_canceled")
plt.title("is_canceled counts")
plt.subplot(1, 2, 2)
hotel_clean["is_canceled"].value_counts().plot.pie(autopct="%1.1f%%")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Categorical / count wale columns: left = count, right = cancel hone ka percentage
cat_cols = ["hotel", "arrival_date_year", "adults", "children", "babies", "meal",
            "market_segment", "distribution_channel", "is_repeated_guest",
            "reserved_room_type", "assigned_room_type", "deposit_type",
            "customer_type", "required_car_parking_spaces", "total_of_special_requests",
            "reservation_status", "total_guests"]

for col in cat_cols:
    order = hotel_clean[col].value_counts().index[:12]
    temp = hotel_clean[hotel_clean[col].isin(order)]
    plt.figure(figsize=(14, 3.8))
    plt.subplot(1, 2, 1)
    sns.countplot(data=temp, y=col, order=order)
    plt.title(col + ": counts")

    plt.subplot(1, 2, 2)
    percent = pd.crosstab(temp[col], temp["is_canceled"], normalize="index") * 100
    percent.loc[order].plot(kind="barh", stacked=True, ax=plt.gca())
    plt.title(col + " vs is_canceled (%)")
    plt.tight_layout()
    plt.show()
    print(hotel_clean[col].value_counts().head(12).to_dict())

print("NOTE: reservation_status booking ke BAAD ka natija hai (Canceled = is_canceled 1), yeh target leakage hai, modelling mein use nahi karna.")

In [ ]:
# Month ke hisaab se bookings aur cancellation rate
plt.figure(figsize=(15, 4))
plt.subplot(1, 2, 1)
sns.countplot(data=hotel_clean, x="arrival_date_month", order=months)
plt.xticks(rotation=60)
plt.title("Bookings per month")
plt.subplot(1, 2, 2)
sns.barplot(data=hotel_clean, x="arrival_date_month", y="is_canceled", order=months)
plt.xticks(rotation=60)
plt.title("Cancellation rate per month")
plt.tight_layout()
plt.show()

In [ ]:
# country, agent aur company
hotel_clean["agent_str"] = hotel_clean["agent"].astype(str)
hotel_clean["has_company"] = (hotel_clean["company"] > 0).astype(int)

for col in ["country", "agent_str", "has_company"]:
    order = hotel_clean[col].value_counts().index[:12]
    temp = hotel_clean[hotel_clean[col].isin(order)]
    plt.figure(figsize=(14, 3.8))
    plt.subplot(1, 2, 1)
    sns.countplot(data=temp, y=col, order=order)
    plt.title(col + ": top 12")
    plt.subplot(1, 2, 2)
    percent = pd.crosstab(temp[col], temp["is_canceled"], normalize="index") * 100
    percent.loc[order].plot(kind="barh", stacked=True, ax=plt.gca())
    plt.title(col + " vs is_canceled (%)")
    plt.tight_layout()
    plt.show()

print("Bina agent ki bookings:", (hotel_clean["agent"] == 0).sum())
print("Company wali bookings:", hotel_clean["has_company"].sum())

In [ ]:
# Numeric columns: histogram | boxplot | is_canceled ke saath boxplot
num_cols = ["lead_time", "arrival_date_week_number", "arrival_date_day_of_month",
            "stays_in_weekend_nights", "stays_in_week_nights", "total_nights",
            "previous_cancellations", "previous_bookings_not_canceled",
            "booking_changes", "days_in_waiting_list", "adr"]

for col in num_cols:
    plt.figure(figsize=(14, 3.5))
    plt.subplot(1, 3, 1)
    sns.histplot(hotel_clean[col], kde=True)
    plt.title(col + ": distribution")
    plt.subplot(1, 3, 2)
    sns.boxplot(x=hotel_clean[col], color="lightcoral")
    plt.title(col + ": boxplot")
    plt.subplot(1, 3, 3)
    sns.boxplot(data=hotel_clean, x="is_canceled", y=col, showfliers=False)
    plt.title(col + " vs is_canceled")
    plt.tight_layout()
    plt.show()
    print(hotel_clean[col].describe().round(2).to_dict())

In [ ]:
# reservation_status_date: har mahine kitni rows aur quarter ke hisaab se cancellation rate
plt.figure(figsize=(15, 3.8))
plt.subplot(1, 2, 1)
hotel_clean.set_index("reservation_status_date").resample("ME").size().plot()
plt.title("Reservation status date: rows per month")
plt.subplot(1, 2, 2)
quarter = hotel_clean["reservation_status_date"].dt.to_period("Q").astype(str)
hotel_clean.groupby(quarter)["is_canceled"].mean().plot(kind="bar")
plt.title("Cancellation rate by quarter")
plt.tight_layout()
plt.show()

# Reserved room aur assigned room ka farq
print("Reserved room != assigned room: %.1f%%" % ((hotel_clean["reserved_room_type"] != hotel_clean["assigned_room_type"]).mean() * 100))

In [ ]:
# Personal columns: name, email, phone-number, credit_card
for col in ["name", "email", "phone-number", "credit_card"]:
    print(col, "-> unique values:", hotel_clean[col].nunique())

plt.figure(figsize=(14, 3.5))
plt.subplot(1, 2, 1)
sns.countplot(data=hotel_clean, y="email_domain", order=hotel_clean["email_domain"].value_counts().index[:8])
plt.title("email domain counts")
plt.subplot(1, 2, 2)
hotel_clean["name"].str.split().str[0].value_counts().head(10).plot(kind="barh")
plt.title("Top 10 first names")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Yeh columns sirf pehchan / private info hain, cancellation samajhne mein kaam ke nahi, is liye privacy ki wajah se drop kiye.")

In [ ]:
# Personal columns drop karna (analysis ho chuka)
hotel_clean = hotel_clean.drop(columns=["name", "email", "phone-number", "credit_card", "email_domain", "agent_str"])
print("Final shape:", hotel_clean.shape)

## 3.6 Correlation aur multivariate

In [ ]:
cols = ["is_canceled", "lead_time", "total_nights", "total_guests", "previous_cancellations",
        "previous_bookings_not_canceled", "booking_changes", "days_in_waiting_list", "adr",
        "required_car_parking_spaces", "total_of_special_requests", "is_repeated_guest"]
plt.figure(figsize=(10, 8))
sns.heatmap(hotel_clean[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap (Hotel)")
plt.show()

In [ ]:
plt.figure(figsize=(15, 4.5))
plt.subplot(1, 2, 1)
adr_table = hotel_clean.pivot_table(index="arrival_date_month", columns="hotel", values="adr", aggfunc="mean").reindex(months)
sns.heatmap(adr_table, annot=True, fmt=".0f", cmap="YlOrRd")
plt.title("Average ADR: month x hotel")
plt.subplot(1, 2, 2)
cancel_table = hotel_clean.pivot_table(index="deposit_type", columns="customer_type", values="is_canceled", aggfunc="mean")
sns.heatmap(cancel_table, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Cancellation rate: deposit_type x customer_type")
plt.tight_layout()
plt.show()

## 3.7 Summary

In [ ]:
print("Rows after cleaning:", len(hotel_clean))
print("Overall cancellation rate: %.1f%%" % (hotel_clean["is_canceled"].mean() * 100))
print("Cancellation % by hotel:")
print((hotel_clean.groupby("hotel")["is_canceled"].mean() * 100).round(1))
print("Cancellation % by deposit_type:")
print((hotel_clean.groupby("deposit_type")["is_canceled"].mean() * 100).round(1))
print("Cancellation % by market_segment:")
print((hotel_clean.groupby("market_segment")["is_canceled"].mean() * 100).round(1))

# Clean data save karna
hotel_clean.to_csv("cleaned_hotel_booking.csv", index=False)
print("Dataset completed successfully!")

**Key findings (Hotel):**
- Missing: `children` (4), `country` (488), `agent` (~14%), `company` (~94%). Agent/company missing ka matlab "involve nahi" tha, is liye 0 bhara.
- Galat rows (0 guests, negative aur 5400 `adr`) hata di; `adr`, `lead_time` aur nights cap kiye. Exact duplicate rows nahi thin.
- Lagbhag 37% bookings cancel hoti hain; City Hotel mein cancellation Resort Hotel se zyada hai.
- Lead time jitna zyada, cancel hone ka chance utna zyada.
- `Non Refund` deposit mein cancellation sab se zyada nazar aati hai (data ka ajeeb pattern).
- Special requests, parking space aur repeated guest hone se cancellation kam hoti hai.
- Peak season (July/August) mein ADR sab se zyada hai.
- `reservation_status` target leakage hai; personal columns privacy ki wajah se drop kiye.

---
# Conclusion
Teeno datasets par missing values, duplicates, data types aur outliers ko handle kiya gaya. Phir har column ka analysis aur graphs banaye gaye, aur har part ke end mein key findings likhi gayi hain.